In [12]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "1212"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#code_files/Grazioso Salvare Logo.png
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('Animal Shelter Database'))),
    html.H2('Created By: Reece Neuenschwander'),
    html.H3('Last Updated: 2/21/2026'),
    html.Center(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))),
    html.Hr(),
    # Radio items for each rescue filter - Reset = Default
    html.Div(
        style={'display' : 'flex'},
            children=[ dcc.RadioItems(
                id='filter-type',
                options=[
                    {'label' : 'Water Rescue', 'value' : 'Water Rescue'},
                    {'label' : 'Mountain Rescue', 'value' : 'Mountain Rescue'},
                    {'label' : 'Disaster Rescue', 'value' : 'Disaster Rescue'},
                    {'label' : 'Reset', 'value' : 'Reset' } 
                ],
                value='Reset',
                labelStyle={'display':'flex'}
            )]

    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'), 
                         row_selectable="single",
                         selected_rows=[0],
                         page_size=10,
                         filter_action="native",
                         sort_action="native",
                         ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
                 
        html.Div(
            id='graph-id',
            className='col s12 m7', # changes to m7 & m5 -- to fit extra table
            style={                 # Style data values to allow both graphs to be displayed side by side
                'display': 'flex',
                'flexDirection': 'row',
                'flexWrap': 'nowrap',        
                'gap': '16px',
                'alignItems': 'stretch',       
                'minHeight': '480px'
            }
            ),
                 
        html.Div(
            id='map-id',
            className='col s12 m5',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    
    # Switch on passed filter type argument value
    match filter_type:
        case 'Water Rescue': # sorts to water rescue criteria
            df = pd.DataFrame.from_records(db.read({"animal_type":"Dog",
                                                    "breed":{"$in":["Labrador Retriever Mix","Chesapeake Bay Retriever","Newfoundland"]},
                                                    "sex_upon_outcome":"Intact Female",
                                                    "age_upon_outcome_in_weeks":{"$gte":26},
                                                    "age_upon_outcome_in_weeks":{"$lte":156}}))
            df.drop(columns=['_id'],inplace=True)

        case 'Mountain Rescue': # sorts to Mountain rescue criteria
            df = pd.DataFrame.from_records(db.read({"animal_type":"Dog",
                                                    "breed":{"$in":["German Shepherd","Alaskan Malamute",
                                                                    "Old English Sheepdog","Siberian Husky",
                                                                   "Rottweiler"]},
                                                    "sex_upon_outcome":"Intact Male",
                                                    "age_upon_outcome_in_weeks":{"$gte":26},
                                                    "age_upon_outcome_in_weeks":{"$lte":156}}))
            df.drop(columns=['_id'],inplace=True)
            
        case 'Disaster Rescue': # sorts to disaster rescue criteria
            df = pd.DataFrame.from_records(db.read({"animal_type":"Dog",
                                                    "breed":{"$in":["Doberman Pinscher"," German Shepherd",
                                                                    "Golden Retriever","Bloodhound",
                                                                   "Rottweiler"]},
                                                    "sex_upon_outcome":"Intact Male",
                                                    "age_upon_outcome_in_weeks":{"$gte":20},
                                                    "age_upon_outcome_in_weeks":{"$lte":300}}))
            df.drop(columns=['_id'],inplace=True)
            
        case _: # Default catches the reset case
            df = pd.DataFrame.from_records(db.read({}))
            df.drop(columns=['_id'],inplace=True)
    
    return df.to_dict('records') # returns the adjusted/sorted data

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    
    # Set default frame
    dff = pd.DataFrame(viewData)
    
    # get data values for table
    breed_data = dff['breed'].value_counts()
    breed_percentage = (breed_data / breed_data.sum() * 100).round(1)
    
    # table data frame
    percent_df = pd.DataFrame({
        'Breed' : breed_data.index,
        'Count' : breed_data.values,
        'Percentage' : breed_percentage.values
    })
    
    # add percent sign to percentage data value
    percent_df['Percentage'] = percent_df['Percentage'].apply(lambda x: f"{x}%") 
    
    # sort the data by percentage
    percent_df = percent_df.sort_values('Count', ascending=False)

    # Percentage table --- Dsiplays the breed, its percentage and its amount next to pie chart
    # is scrollable
    percentage_table = dash_table.DataTable(
        id='breed-percent-table',
        data=percent_df.to_dict('records'),
        columns=[
            {"name": "Breed", "id": "Breed"},
            {"name": "Count", "id" : "Count"},
            {"name": "Percentage", "id": "Percentage"}
        ],
        style_table={
            'height': '400px',           
            'overflowY': 'auto',         
            'border': '1px solid #ddd',
            'borderRadius': '6px',
        },
        style_cell={
            'textAlign': 'left',
            'padding': '8px',
        },
        style_header={
            'backgroundColor': '#f4f4f4',
            'fontWeight': 'bold',
            'borderBottom': '2px solid #aaa',
        },
        style_data_conditional=[
            {'if': {'row_index': 'odd'}, 'backgroundColor': '#f9f9f9'},
        ], 
    )
    
    # Pie chart --- Displays breed percentages and names in pie chart format
    chart = px.pie(
        dff,
        names='breed',
        title='Breed Distribution',
    )
    chart.update_traces(textposition='inside',textinfo='percent+label')
    
    return [ # Returns the table and then pie chart --- together = same row
        
        html.Div(
            [
                html.H4("Breed Details",
                        style={'margin':'0 0 10px 0', 'textAlign': 'center'}),
                percentage_table
            ],
            style={
                'padding':'10px',
                'borderRadius':'8px'
            }
        ),
        dcc.Graph(figure=chart, style={'flex':'1','minWidth':'0','height':'450px'}) # flex style to fit both graphs
    ]

    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    
      # Returns blank map on application start before default selection is recognized
    if dff.empty:
        return [
            dl.Map(style={'width': '1880px', 'height': '500px'},  # extended to fill under table
            center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id")])
        ]
    
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48] # 1000px width
    return [
        dl.Map(style={'width': '750px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://targetbaggage-timegranite-3000.codio.io/proxy/8050/
